# Inference: Score and Rank Customers

Load the trained model, score new customers, and generate a retention target list.


In [9]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root (folder containing src/) to path
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))

print("CWD:", cwd)
print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())

from src.predict import load_model, score_and_rank, export_for_retention_team, estimate_intervention_cost
from src.data_prep import parse_income
from src.features import (
    engineer_account_age,
    engineer_financial_ratios,
    engineer_engagement_ratios,
    engineer_engagement_flags
)

print("Modules loaded")

CWD: d:\Bank-churn-ml\notebooks
Project root: d:\Bank-churn-ml
src exists: True
Modules loaded


## 1. Load cleaned data and model


In [10]:
# Load data (use cleaned version from pipeline)
df = pd.read_csv('../data/interim/bank_churn_cleaned.csv')
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
display(df.head(2))

Data shape: (10299, 23)

Columns: ['customer_id', 'full_name', 'age', 'gender', 'region', 'account_type', 'account_open_date', 'tenure_years', 'credit_score', 'annual_income', 'account_balance', 'num_products', 'monthly_transactions', 'complaints_12mo', 'debt_to_income', 'digital_engagement', 'has_active_loan', 'has_credit_card', 'churned', 'annual_income_cleaned', 'complaints_12mo_missing', 'credit_score_missing', 'gender_missing']


,customer_id,full_name,age,gender,region,account_type,account_open_date,tenure_years,credit_score,annual_income,...,complaints_12mo,debt_to_income,digital_engagement,has_active_loan,has_credit_card,churned,annual_income_cleaned,complaints_12mo_missing,credit_score_missing,gender_missing
0,CUST-007684,Eric Flores,62,Female,Metro Manila,Checking,1990-01-01,2.2,684.0,38405,...,3.0,0.339,61.6,0,0.0,0,38405.0,0,1,0
1,CUST-003889,Mary Peterson,43,Female,Cebu,Business,1990-01-02,5.1,843.0,48383,...,0.0,0.258,71.2,0,1.0,0,48383.0,0,0,0


In [11]:
# Load trained model
model = load_model('../models/best_churn_model.pkl')
print(f"Model loaded: {model}")

Model loaded: Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'tenure_years',
                                                   'credit_score',
                                                   'account_balance',
                                                   'num_products',
                                                   'monthly_transactions',
                                                   'complaints_12mo',
                                                   'debt_to_income',
                                                   'digital_engagem

## 2. Engineer features (same as training)


In [12]:
df_scored = df.copy()

# Apply same feature engineering as training
df_scored['account_open_date'] = pd.to_datetime(df_scored['account_open_date'], errors='coerce')
ref_date = pd.Timestamp('2024-01-01')
df_scored['account_age_days'] = (ref_date - df_scored['account_open_date']).dt.days

eps = 1e-6
df_scored['balance_income_ratio'] = df_scored['account_balance'] / (df_scored['annual_income_cleaned'] + eps)
df_scored['txn_per_product'] = df_scored['monthly_transactions'] / (df_scored['num_products'] + 1.0)
df_scored['complaint_txn_ratio'] = df_scored['complaints_12mo'] / (df_scored['monthly_transactions'] + 1.0)
df_scored['low_engagement_flag'] = (df_scored['digital_engagement'] < 40).astype(int)

print("Features engineered")
print(f"Final feature shape: {df_scored.shape}")

Features engineered
Final feature shape: (10299, 28)


## 3. Prepare features for scoring


In [13]:
# Drop same columns as training
drop_cols = ['churned', 'customer_id', 'full_name', 'annual_income', 'account_open_date']

X_score = df_scored.drop(columns=[c for c in drop_cols if c in df_scored.columns], errors='ignore')
customer_ids = df_scored['customer_id']

print(f"Feature matrix shape: {X_score.shape}")
print(f"Features: {X_score.columns.tolist()}")

Feature matrix shape: (10299, 23)
Features: ['age', 'gender', 'region', 'account_type', 'tenure_years', 'credit_score', 'account_balance', 'num_products', 'monthly_transactions', 'complaints_12mo', 'debt_to_income', 'digital_engagement', 'has_active_loan', 'has_credit_card', 'annual_income_cleaned', 'complaints_12mo_missing', 'credit_score_missing', 'gender_missing', 'account_age_days', 'balance_income_ratio', 'txn_per_product', 'complaint_txn_ratio', 'low_engagement_flag']


## 4. Score all customers


In [14]:
# Score customers
THRESHOLD = 0.34  # From validation

results = score_and_rank(model, X_score, customer_ids=customer_ids, threshold=THRESHOLD)

print(f"Scored {len(results)} customers")
display(results.head(10))

Scored 10299 customers


,customer_id,churn_probability,decision,rank,recommended_action
0,CUST-DUP-44387,0.995048,1,1.0,Outreach - high churn risk
1,CUST-008202,0.995048,1,1.0,Outreach - high churn risk
2,CUST-000053,0.993539,1,2.0,Outreach - high churn risk
3,CUST-DUP-51555,0.993539,1,2.0,Outreach - high churn risk
4,CUST-DUP-11610,0.993409,1,3.0,Outreach - high churn risk
5,CUST-000080,0.993409,1,3.0,Outreach - high churn risk
6,CUST-009234,0.983414,1,4.0,Outreach - high churn risk
7,CUST-002422,0.981645,1,5.0,Outreach - high churn risk
8,CUST-DUP-90079,0.981386,1,6.0,Outreach - high churn risk
9,CUST-008200,0.980737,1,7.0,Outreach - high churn risk


## 5. Summary statistics


In [15]:
flagged_count = (results['decision'] == 1).sum()
flagged_rate = flagged_count / len(results)

print(f"Total customers scored: {len(results)}")
print(f"Flagged for intervention: {flagged_count} ({flagged_rate*100:.1f}%)")
print(f"\nChurn probability statistics:")
print(results['churn_probability'].describe())
print(f"\nFlagged customers (top 10 by risk):")
display(results[results['decision'] == 1].head(10))

Total customers scored: 10299
Flagged for intervention: 153 (1.5%)

Churn probability statistics:
count    10299.000000
mean         0.044994
std          0.114943
min          0.000000
25%          0.009302
50%          0.018995
75%          0.039352
max          0.995048
Name: churn_probability, dtype: float64

Flagged customers (top 10 by risk):


,customer_id,churn_probability,decision,rank,recommended_action
0,CUST-DUP-44387,0.995048,1,1.0,Outreach - high churn risk
1,CUST-008202,0.995048,1,1.0,Outreach - high churn risk
2,CUST-000053,0.993539,1,2.0,Outreach - high churn risk
3,CUST-DUP-51555,0.993539,1,2.0,Outreach - high churn risk
4,CUST-DUP-11610,0.993409,1,3.0,Outreach - high churn risk
5,CUST-000080,0.993409,1,3.0,Outreach - high churn risk
6,CUST-009234,0.983414,1,4.0,Outreach - high churn risk
7,CUST-002422,0.981645,1,5.0,Outreach - high churn risk
8,CUST-DUP-90079,0.981386,1,6.0,Outreach - high churn risk
9,CUST-008200,0.980737,1,7.0,Outreach - high churn risk


## 6. Export for retention team


In [16]:
# Export top 200 at-risk customers
retention_targets = export_for_retention_team(results, output_path='../retention_targets.csv', top_n=200)

print(f"\nPriority breakdown:")
print(retention_targets['engagement_priority'].value_counts())

✅ Exported 153 customers to ../retention_targets.csv

Priority breakdown:
engagement_priority
Urgent      136
Standard     10
High          7
Name: count, dtype: int64


## 7. Campaign cost/benefit analysis


In [17]:
metrics = estimate_intervention_cost(
    results,
    cost_per_call=5,              # $ per outreach
    cost_per_lost_customer=500    # $ lost revenue per churner
)

print("\n" + "="*60)
print("CAMPAIGN COST/BENEFIT ANALYSIS")
print("="*60)
print(f"Customers to contact:        {metrics['customers_flagged']:,}")
print(f"Estimated churners (untreated): {int(metrics['estimated_churners_if_notreated']):,}")
print(f"\nCampaign cost (@ $5/call):   ${metrics['campaign_cost']:,.0f}")
print(f"Expected savings:             ${metrics['expected_savings']:,.0f}")
print(f"Net benefit:                  ${metrics['net_benefit']:,.0f}")
print(f"ROI:                          {metrics['roi']:.2f}x")
print("="*60)


CAMPAIGN COST/BENEFIT ANALYSIS
Customers to contact:        153
Estimated churners (untreated): 138

Campaign cost (@ $5/call):   $765
Expected savings:             $69,402
Net benefit:                  $68,637
ROI:                          89.72x
